In [4]:
import pandas as pd
import os

# Definujeme cesty
RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

def safe_convert_to_parquet(directory):
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            csv_path = os.path.join(directory, filename)
            parquet_path = csv_path.replace('.csv', '.parquet')

            print(f"Zpracovávám: {filename}")

            try:
                # --- LOGIKA PRO REGISTRY (Windows-1250, středník) ---
                if 'export_sluzby' in filename or 'czech_registre' in filename:
                    # Registry z NRPZS vyžadují specifické nastavení
                    df = pd.read_csv(csv_path, sep=';', encoding='cp1250', low_memory=False)

                # --- LOGIKA PRO OSTATNÍ (UTF-8, čárka) ---
                else:
                    # Ostatní soubory (jako medical_data.csv) jsou standardní
                    df = pd.read_csv(csv_path, encoding='utf-8')

                # Uložení do Parquet (tento formát už interně používá UTF-8 a je zkomprimovaný)
                df.to_parquet(parquet_path, index=False)

                old_size = os.path.getsize(csv_path) / (1024 * 1024)
                new_size = os.path.getsize(parquet_path) / (1024 * 1024)
                print(f"Převedeno: {old_size:.1f} MB -> {new_size:.1f} MB")

            except Exception as e:
                print(f"Chyba při zpracování {filename}: {e}")

# Spustíme převod
print("Převádím soubory v data/raw...")
safe_convert_to_parquet(RAW_DIR)

print("\nPřevádím soubory v data/processed...")
safe_convert_to_parquet(PROCESSED_DIR)

Převádím soubory v data/raw...
Zpracovávám: export_2026.csv
Chyba při zpracování export_2026.csv: Error tokenizing data. C error: Expected 5 fields in line 12, saw 23

Zpracovávám: export_sluzby_2026.csv
Převedeno: 40.8 MB -> 5.5 MB
Zpracovávám: medical_transcriptions.csv
Převedeno: 16.2 MB -> 7.7 MB

Převádím soubory v data/processed...
Zpracovávám: czech_registre_final.csv
Chyba při zpracování czech_registre_final.csv: 'charmap' codec can't decode byte 0x88 in position 2421: character maps to <undefined>
Zpracovávám: czech_registre_final_small.csv
Chyba při zpracování czech_registre_final_small.csv: 'charmap' codec can't decode byte 0x88 in position 1180: character maps to <undefined>
Zpracovávám: kaggle_final.csv
Převedeno: 11.7 MB -> 5.7 MB
